# Few-run conditional surrogates: Fisher-isometric conditioning on Hasegawa–Wakatani (Colab)

This notebook runs the Tier-1 testbed of `docs/plasma_surrogate_conditioning.md` (§8.1). It is self-contained: the cells below write the package, so no repository checkout is needed.

**Testbed.** A batched pseudo-spectral solver for the modified Hasegawa–Wakatani model on a doubly periodic $64^2$ grid, with $d=5$ control parameters
$\theta=(\log_{10}\alpha,\ \kappa,\ \log_{10}\hat\nu,\ \log_{10}\hat D,\ k_0)$: adiabaticity, gradient drive, grid-relative hyperviscosity and hyperdiffusion, and box wavenumber.
- One surrogate step is $\Delta t=1$ (50 solver steps), taken after a spin-up of $t=250$, with 120 snapshots per run.
- **Branch restarts** from stored states at $\theta\pm\varepsilon e_i$ give the *true* one-step sensitivity $\partial s_{t+1}/\partial\theta$, which is used for Jacobian supervision and for the sensitivity metrics.
- Viscosity and diffusion are sloppy directions (one-step sensitivity 10–100× smaller); α, κ and $k_0$ are stiff.

**Surrogate.** A patch transformer (circular-conv stem + patchify → adaLN-Zero blocks → unpatchify + conv refinement). The conditioning $c=g(\theta)$ enters the tokens, every block and the decoder. States are divided by their own per-field RMS, the target is $s_{t+1}/\mathrm{rms}(s_t)$, and $\log\mathrm{rms}$ is passed as extra conditioning. The output metric is therefore a fixed relative-error metric ($W=I$ in that frame).

| arm | conditioning |
|---|---|
| `A0_film` | FiLM/adaLN on raw standardised θ (baseline) |
| `A1_g` | learned $g$, weight decay on the conditioning path |
| `A2_iso` | A1 + $\mathcal R_{\rm iso}$ against the surrogate's own Fisher $A_{\rm self}$ |
| `A3_iso_wn` | A2 + Fisher-whitened θ-noise |
| `A4_iso_jac` | A2 + Jacobian supervision from branch restarts, with the physical Fisher $A_{\rm phys}$ as target |
| `A5_iso_in` | A2 + isotropic θ-noise of matched magnitude (control for A3) |
| `A6_full` | A3 + Jacobian supervision + $A_{\rm phys}$ target |

The **decisive** first comparison is A0 / A1 / A2 / A4. A3 vs A5 isolates the whitening, and A6 is the full method.

All splits are by run: the validation runs (checkpoint selection) and the 48 test runs are disjoint from the training pool and fixed across seeds and budgets. Budgets are nested prefixes of one permutation of the pool per seed.

**Use a GPU runtime** (Runtime → Change runtime type). An A100 or L4 is best; a T4 works but is slower.

In [ ]:
import os, sys, torch
from pathlib import Path
WORK = Path('/content/unified_sbi_surrogate')
for sub in ('src/unified_sbi/surrogate', 'scripts', 'tests'):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
os.chdir(WORK); sys.path.insert(0, str(WORK / 'src'))
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 1. Package source
The next cells write the package exactly as it is in the repository (`src/unified_sbi/surrogate/`). Expand them to read the code.

In [ ]:
%%writefile src/unified_sbi/__init__.py
"""Standalone copy for Colab: only the surrogate subpackage."""


In [ ]:
%%writefile src/unified_sbi/surrogate/__init__.py
"""Few-run conditional surrogates with Fisher-isometric conditioning
(see docs/plasma_surrogate_conditioning.md)."""


In [ ]:
%%writefile src/unified_sbi/surrogate/hw.py
"""Batched pseudo-spectral solver for the modified Hasegawa-Wakatani (HW) model, in torch.

This is the Tier-1 testbed of ``docs/plasma_surrogate_conditioning.md`` (section 8.1): a cheap
reduced plasma-turbulence model with ``d = 5`` control parameters, any number of runs, any
design, and branch restarts that give the *true* one-step sensitivity ds_{t+1}/dtheta.

Equations (x radial, y poloidal; zonal = ky == 0 modes; tilde = non-zonal part):

    d_t Omega + [phi, Omega] = alpha (phi~ - n~)                - nu k^{2p} Omega
    d_t n     + [phi, n]     = alpha (phi~ - n~) - kappa d_y phi - D  k^{2p} n
    Omega = lap phi,          [a, b] = d_x a d_y b - d_y a d_x b

on a doubly periodic box of side L = 2 pi / k0 with ``n x n`` points. Time stepping is RK4
with an integrating factor for the (diagonal) hyperdiffusion, 2/3-rule dealiasing of the
bracket, and fixed ``dt``. Every run in a batch has its own parameters.

Parameters (natural coordinates, uniform prior on a box; ``standardise`` maps the box to zero
mean and unit variance per coordinate):

    theta = (log10 alpha, kappa, log10 nu_hat, log10 D_hat, k0)

The hyperdiffusion coefficients are grid-relative: nu_hat and D_hat are the damping rates at
the dealiased cutoff k_c = k0 n / 3, i.e. nu = nu_hat / k_c^(2p). This keeps grid-scale
dissipation in a controlled range for every box size.

The surrogate's state is the pair of real fields (n, phi). A restart from a stored (n, phi)
rebuilds Omega = lap phi with the *restart's* k0, so the one-step map (state array, theta) ->
next state array is well defined for every theta, which is exactly what the surrogate learns.
"""

from __future__ import annotations

import math
from dataclasses import asdict, dataclass, field

import numpy as np
import torch

NAMES = ("log10_alpha", "kappa", "log10_nu_hat", "log10_D_hat", "k0")


@dataclass
class HWConfig:
    n: int = 64                    # grid points per side
    p: int = 3                     # hyperdiffusion order: damping nu k^(2p)
    dt: float = 0.02               # solver step
    save_dt: float = 1.0           # time between stored snapshots (= one surrogate step)
    t_spin: float = 250.0          # spin-up before the first stored snapshot
    n_save: int = 120              # stored snapshots per run
    ic_amp: float = 0.2            # amplitude of the random initial condition
    batch: int = 64                # runs integrated together
    # prior box in natural coordinates (see NAMES)
    lo: tuple = (-1.0, 0.7, math.log10(0.3), math.log10(0.3), 0.15)
    hi: tuple = (0.0, 1.4, 1.0, 1.0, 0.25)

    @property
    def d(self):
        return len(self.lo)

    @property
    def steps_per_save(self):
        return int(round(self.save_dt / self.dt))

    def to_dict(self):
        return asdict(self)


# ---------------------------------------------------------------------------------------------
# parameters


def box(cfg: HWConfig, dtype=torch.float64):
    return torch.tensor(cfg.lo, dtype=dtype), torch.tensor(cfg.hi, dtype=dtype)


def standardise(theta, cfg: HWConfig):
    """Natural coordinates -> zero mean, unit variance under the uniform prior."""
    lo, hi = box(cfg, theta.dtype)
    lo, hi = lo.to(theta.device), hi.to(theta.device)
    return (theta - (lo + hi) / 2) / ((hi - lo) / 2 / math.sqrt(3))


def unstandardise(u, cfg: HWConfig):
    lo, hi = box(cfg, u.dtype)
    lo, hi = lo.to(u.device), hi.to(u.device)
    return u * ((hi - lo) / 2 / math.sqrt(3)) + (lo + hi) / 2


def sample_prior(n, cfg: HWConfig, rng: np.random.Generator):
    lo, hi = np.array(cfg.lo), np.array(cfg.hi)
    return lo + (hi - lo) * rng.random((n, cfg.d))


def latin_hypercube(n, cfg: HWConfig, rng: np.random.Generator):
    lo, hi = np.array(cfg.lo), np.array(cfg.hi)
    u = (np.stack([rng.permutation(n) for _ in range(cfg.d)], 1) + rng.random((n, cfg.d))) / n
    return lo + (hi - lo) * u


def physical(theta, cfg: HWConfig):
    """(B, 5) natural coordinates -> dict of (B, 1, 1) physical parameters."""
    th = theta[:, :, None, None]
    kc = (th[:, 4] * cfg.n / 3.0) ** (2 * cfg.p)
    return dict(alpha=10 ** th[:, 0], kappa=th[:, 1], nu=10 ** th[:, 2] / kc, D=10 ** th[:, 3] / kc,
                k0=th[:, 4])


# ---------------------------------------------------------------------------------------------
# spectral operators


@dataclass
class Grid:
    kx: torch.Tensor      # (B, n, 1)
    ky: torch.Tensor      # (B, 1, n//2+1)
    k2: torch.Tensor      # (B, n, n//2+1)
    inv_k2: torch.Tensor
    dealias: torch.Tensor  # (n, n//2+1) bool
    nonzonal: torch.Tensor  # (1, n//2+1) real
    n: int = field(default=0)


def make_grid(k0, n, device, dtype):
    """k0: (B, 1, 1). Wavenumbers are k0 * integer."""
    ix = torch.fft.fftfreq(n, d=1.0 / n, device=device, dtype=dtype)       # integers
    iy = torch.arange(n // 2 + 1, device=device, dtype=dtype)
    kx = k0 * ix[None, :, None]
    ky = k0 * iy[None, None, :]
    k2 = kx ** 2 + ky ** 2
    inv_k2 = torch.where(k2 > 0, 1.0 / k2.clamp_min(1e-30), torch.zeros_like(k2))
    cut = n / 3.0
    dealias = (ix.abs()[:, None] < cut) & (iy[None, :] < cut)
    nonzonal = (iy > 0).to(dtype)[None, :]
    return Grid(kx, ky, k2, inv_k2, dealias, nonzonal, n)


def _deriv(fh, g: Grid):
    """Real-space d_x f, d_y f from the spectrum fh."""
    n = g.n
    fx = torch.fft.irfft2(1j * g.kx * fh, s=(n, n))
    fy = torch.fft.irfft2(1j * g.ky * fh, s=(n, n))
    return fx, fy


def rhs(nh, wh, par, g: Grid):
    """Nonlinear + coupling + drive tendencies (hyperdiffusion handled by the integrating factor)."""
    ph = -wh * g.inv_k2
    phx, phy = _deriv(ph * g.dealias, g)
    nx, ny = _deriv(nh * g.dealias, g)
    wx, wy = _deriv(wh * g.dealias, g)
    br_n = torch.fft.rfft2(phx * ny - phy * nx) * g.dealias
    br_w = torch.fft.rfft2(phx * wy - phy * wx) * g.dealias
    coup = par["alpha"] * (ph - nh) * g.nonzonal
    dn = coup - br_n - par["kappa"] * 1j * g.ky * ph
    dw = coup - br_w
    return dn, dw


class HWState:
    """Spectral state for a batch of runs with per-run parameters."""

    def __init__(self, theta, cfg: HWConfig, device="cpu", dtype=torch.float32):
        self.cfg, self.dtype, self.device = cfg, dtype, device
        theta = torch.as_tensor(theta, dtype=dtype, device=device)
        self.theta = theta
        self.par = physical(theta, cfg)
        self.g = make_grid(self.par["k0"], cfg.n, device, dtype)
        kp = self.g.k2 ** cfg.p
        h = cfg.dt
        self.E_n = torch.exp(-self.par["D"] * kp * h / 2)      # half-step integrating factors
        self.E_w = torch.exp(-self.par["nu"] * kp * h / 2)
        self.nh = self.wh = None

    # --- initial conditions and conversions -------------------------------------------------
    def random_ic(self, seeds):
        n, B = self.cfg.n, self.theta.shape[0]
        fields = []
        for s in seeds:
            gen = torch.Generator().manual_seed(int(s))
            fields.append(self.cfg.ic_amp * torch.randn(2, n, n, generator=gen, dtype=torch.float64))
        f = torch.stack(fields).to(self.device, self.dtype)
        f = f - f.mean((-2, -1), keepdim=True)
        self.nh = torch.fft.rfft2(f[:, 0]) * self.g.dealias
        self.wh = torch.fft.rfft2(f[:, 1]) * self.g.dealias
        assert self.nh.shape[0] == B
        return self

    def set_fields(self, n_phi):
        """n_phi: (B, 2, n, n) real fields (n, phi) -> spectral (n, Omega) with this batch's k0."""
        f = torch.as_tensor(n_phi, device=self.device, dtype=self.dtype)
        self.nh = torch.fft.rfft2(f[:, 0])
        self.wh = -self.g.k2 * torch.fft.rfft2(f[:, 1])
        return self

    def fields(self):
        """(B, 2, n, n) real (n, phi)."""
        n = self.cfg.n
        ph = -self.wh * self.g.inv_k2
        return torch.stack([torch.fft.irfft2(self.nh, s=(n, n)), torch.fft.irfft2(ph, s=(n, n))], 1)

    # --- time stepping ----------------------------------------------------------------------
    def step(self):
        """One IF-RK4 step of size dt."""
        h, En, Ew, par, g = self.cfg.dt, self.E_n, self.E_w, self.par, self.g
        n0, w0 = self.nh, self.wh
        a_n, a_w = rhs(n0, w0, par, g)
        n1, w1 = En * (n0 + h / 2 * a_n), Ew * (w0 + h / 2 * a_w)
        b_n, b_w = rhs(n1, w1, par, g)
        n2, w2 = En * n0 + h / 2 * b_n, Ew * w0 + h / 2 * b_w
        c_n, c_w = rhs(n2, w2, par, g)
        n3, w3 = En * (En * n0 + h * c_n), Ew * (Ew * w0 + h * c_w)
        d_n, d_w = rhs(n3, w3, par, g)
        self.nh = En * En * n0 + h / 6 * (En * En * a_n + 2 * En * (b_n + c_n) + d_n)
        self.wh = Ew * Ew * w0 + h / 6 * (Ew * Ew * a_w + 2 * Ew * (b_w + c_w) + d_w)
        return self

    def advance(self, n_steps):
        for _ in range(n_steps):
            self.step()
        return self

    def finite(self):
        return torch.isfinite(self.nh).all((-2, -1)) & torch.isfinite(self.wh).all((-2, -1))


# ---------------------------------------------------------------------------------------------
# diagnostics


def flux(n_phi, k0):
    """Radial particle flux Gamma_n = -< n d_y phi > per field. n_phi: (..., 2, n, n); k0: (...)."""
    f = torch.as_tensor(n_phi)
    k0 = torch.as_tensor(k0, dtype=f.dtype, device=f.device)
    n = f.shape[-1]
    ky = torch.arange(n // 2 + 1, device=f.device, dtype=f.dtype)
    phy = torch.fft.irfft(1j * ky * torch.fft.rfft(f[..., 1, :, :], dim=-1), n=n, dim=-1)
    return -(f[..., 0, :, :] * phy).mean((-2, -1)) * k0


def energy(n_phi, k0):
    """E = 1/2 < n^2 + |grad phi|^2 >."""
    f = torch.as_tensor(n_phi)
    k0 = torch.as_tensor(k0, dtype=f.dtype, device=f.device)
    n = f.shape[-1]
    ph = torch.fft.rfft2(f[..., 1, :, :])
    ix = torch.fft.fftfreq(n, d=1.0 / n, device=f.device, dtype=f.dtype)
    iy = torch.arange(n // 2 + 1, device=f.device, dtype=f.dtype)
    gx = torch.fft.irfft2(1j * ix[:, None] * ph, s=(n, n))
    gy = torch.fft.irfft2(1j * iy[None, :] * ph, s=(n, n))
    k0 = k0[..., None, None]
    return 0.5 * (f[..., 0, :, :] ** 2 + k0 ** 2 * (gx ** 2 + gy ** 2)).mean((-2, -1))


# ---------------------------------------------------------------------------------------------
# runs and branch restarts


def simulate(theta, seeds, cfg: HWConfig, device="cpu", dtype=torch.float32, log=None,
             record_spinup_every=0):
    """Integrate runs from random initial conditions.

    Returns a dict with ``states`` (R, n_save, 2, n, n) float32 (n, phi) at times
    t_spin + j * save_dt, ``flux`` (R, n_save), ``finite`` (R,) bool, and optionally the energy
    during spin-up (to check saturation)."""
    theta = np.asarray(theta, dtype=np.float64)
    R = len(theta)
    out = torch.zeros(R, cfg.n_save, 2, cfg.n, cfg.n, dtype=torch.float32)
    fl = torch.zeros(R, cfg.n_save)
    ok = torch.ones(R, dtype=torch.bool)
    spin = []
    spin_steps = int(round(cfg.t_spin / cfg.dt))
    for b0 in range(0, R, cfg.batch):
        sl = slice(b0, min(R, b0 + cfg.batch))
        st = HWState(theta[sl], cfg, device, dtype).random_ic(seeds[sl])
        k0 = st.par["k0"].reshape(-1)
        es = []
        if record_spinup_every:
            for i in range(0, spin_steps, record_spinup_every):
                st.advance(min(record_spinup_every, spin_steps - i))
                es.append(energy(st.fields(), k0).cpu())
            spin.append(torch.stack(es, 1))
        else:
            st.advance(spin_steps)
        for j in range(cfg.n_save):
            if j:
                st.advance(cfg.steps_per_save)
            f = st.fields()
            out[sl, j] = f.float().cpu()
            fl[sl, j] = flux(f, k0).float().cpu()
        ok[sl] = st.finite().cpu() & torch.isfinite(out[sl]).flatten(1).all(1)
        if log:
            log(f"  simulated runs {sl.start}-{sl.stop - 1} / {R}")
    res = dict(states=out, flux=fl, finite=ok)
    if record_spinup_every:
        res["spinup_energy"] = torch.cat(spin)
    return res


def branch_jacobian(states, theta, cfg: HWConfig, eps=0.02, device="cpu", dtype=torch.float64,
                    batch=None):
    """True one-step sensitivity by central differences of branch restarts.

    states: (S, 2, n, n) stored (n, phi); theta: (S, d) natural coordinates.
    ``eps`` is in *standardised* units. Returns J (S, d, 2, n, n) = d s_{t+1} / d theta_std and
    the unperturbed next state (S, 2, n, n)."""
    states = torch.as_tensor(states)
    th = torch.as_tensor(np.asarray(theta), dtype=torch.float64)
    S, d = th.shape
    scale = ((torch.tensor(cfg.hi) - torch.tensor(cfg.lo)) / 2 / math.sqrt(3)).double()
    # rows: base, then +e_i, -e_i for each i
    pert = [th]
    for i in range(d):
        for sgn in (1, -1):
            t = th.clone()
            t[:, i] += sgn * eps * scale[i]
            pert.append(t)
    P = torch.stack(pert, 1).reshape(-1, d)                      # (S*(2d+1), d)
    X = states[:, None].expand(S, 2 * d + 1, *states.shape[1:]).reshape(-1, *states.shape[1:])
    batch = batch or 4 * cfg.batch
    Y = torch.empty(X.shape, dtype=torch.float64)
    for b0 in range(0, len(P), batch):
        sl = slice(b0, min(len(P), b0 + batch))
        st = HWState(P[sl].numpy(), cfg, device, dtype).set_fields(X[sl].to(dtype))
        st.advance(cfg.steps_per_save)
        Y[sl] = st.fields().cpu().double()
    Y = Y.reshape(S, 2 * d + 1, *states.shape[1:])
    J = (Y[:, 1::2] - Y[:, 2::2]) / (2 * eps)
    return J.float(), Y[:, 0].float()


In [ ]:
%%writefile src/unified_sbi/surrogate/nets.py
"""Networks: the parameter embedding g and a conditional patch-transformer surrogate.

The surrogate mirrors the production layout (grid2mesh encoder -> attention processors -> z ->
mesh2grid decoder) at Colab scale:

    x_t (2 x n x n, instance-normalised) -> circular conv + patchify (+ c)       "grid2mesh"
      -> L transformer blocks with adaLN-Zero conditioning on c                  "processors"
      -> final adaLN + linear unpatchify + circular conv refinement              "mesh2grid"
      -> residual prediction  y_hat = x_t + delta

The conditioning vector enters (i) every token at the encoder, (ii) every processor block and
(iii) the decoder, as in section 2 of docs/plasma_surrogate_conditioning.md. Everything that
reads the conditioning (g, the conditioning MLP and all modulation projections) is exposed by
``conditioning_parameters`` so that it can get its own weight decay.

Scale handling: the state is divided by its own per-field RMS r_t (a property of the input,
not of theta), the target is s_{t+1} / r_t, and log r_t is appended to the conditioning so the
network still knows the absolute amplitude (which sets the strength of the nonlinearity).
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn


class ParamEmbed(nn.Module):
    """g: standardised theta (d) -> c (K). ``identity=True`` gives the raw-theta baseline."""

    def __init__(self, d, K=8, hidden=128, depth=2, identity=False):
        super().__init__()
        self.identity, self.d = identity, d
        self.K = d if identity else K
        if not identity:
            layers, w = [], d
            for _ in range(depth):
                layers += [nn.Linear(w, hidden), nn.SiLU()]
                w = hidden
            layers.append(nn.Linear(w, K))
            self.net = nn.Sequential(*layers)

    def forward(self, theta):
        return theta if self.identity else self.net(theta)


class Block(nn.Module):
    """DiT-style transformer block with adaLN-Zero modulation from the conditioning vector."""

    def __init__(self, w, heads, mlp_ratio=4.0):
        super().__init__()
        self.heads = heads
        self.n1 = nn.LayerNorm(w, elementwise_affine=False, eps=1e-6)
        self.n2 = nn.LayerNorm(w, elementwise_affine=False, eps=1e-6)
        self.qkv = nn.Linear(w, 3 * w)
        self.proj = nn.Linear(w, w)
        h = int(w * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(w, h), nn.GELU(approximate="tanh"), nn.Linear(h, w))
        self.mod = nn.Linear(w, 6 * w)
        nn.init.zeros_(self.mod.weight); nn.init.zeros_(self.mod.bias)

    def forward(self, x, cvec):
        B, T, w = x.shape
        s1, b1, g1, s2, b2, g2 = self.mod(F.silu(cvec))[:, None].chunk(6, -1)
        h = self.n1(x) * (1 + s1) + b1
        q, k, v = self.qkv(h).reshape(B, T, 3, self.heads, w // self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, T, w)
        x = x + g1 * self.proj(a)
        h = self.n2(x) * (1 + s2) + b2
        return x + g2 * self.mlp(h)


class Surrogate(nn.Module):
    def __init__(self, n=64, fields=2, c_dim=8, width=192, depth=6, heads=6, patch=4,
                 extra_cond=2):
        super().__init__()
        assert n % patch == 0 and width % heads == 0
        self.n, self.f, self.p, self.w = n, fields, patch, width
        self.T = (n // patch) ** 2
        # conditioning: [c, log r_t] -> cvec
        self.cond = nn.Sequential(nn.Linear(c_dim + extra_cond, width), nn.SiLU(),
                                  nn.Linear(width, width))
        # grid2mesh
        self.stem = nn.Conv2d(fields, width // 4, 3, padding=1, padding_mode="circular")
        self.patchify = nn.Conv2d(width // 4, width, patch, stride=patch)
        self.pos = nn.Parameter(torch.randn(1, self.T, width) * 0.02)
        self.tok_cond = nn.Linear(width, width)
        # processors
        self.blocks = nn.ModuleList([Block(width, heads) for _ in range(depth)])
        # mesh2grid
        self.n_out = nn.LayerNorm(width, elementwise_affine=False, eps=1e-6)
        self.mod_out = nn.Linear(width, 2 * width)
        nn.init.zeros_(self.mod_out.weight); nn.init.zeros_(self.mod_out.bias)
        self.unpatch = nn.Linear(width, patch * patch * fields)
        self.refine = nn.Sequential(
            nn.Conv2d(2 * fields, 32, 3, padding=1, padding_mode="circular"), nn.GELU(),
            nn.Conv2d(32, fields, 3, padding=1, padding_mode="circular"))
        nn.init.zeros_(self.refine[-1].weight); nn.init.zeros_(self.refine[-1].bias)

    def conditioning_parameters(self):
        mods = [self.cond, self.tok_cond, self.mod_out] + [b.mod for b in self.blocks]
        return [p for m in mods for p in m.parameters()]

    def forward(self, x, c, logr):
        """x: (B, f, n, n) normalised state; c: (B, c_dim); logr: (B, extra_cond)."""
        B = x.shape[0]
        cvec = self.cond(torch.cat([c, logr], -1))
        h = self.patchify(F.gelu(self.stem(x))).flatten(2).transpose(1, 2)    # (B, T, w)
        h = h + self.pos + self.tok_cond(F.silu(cvec))[:, None]
        for blk in self.blocks:
            h = blk(h, cvec)
        s, b = self.mod_out(F.silu(cvec))[:, None].chunk(2, -1)
        h = self.unpatch(self.n_out(h) * (1 + s) + b)                          # (B, T, p*p*f)
        g = self.n // self.p
        d = h.reshape(B, g, g, self.p, self.p, self.f).permute(0, 5, 1, 3, 2, 4)
        d = d.reshape(B, self.f, self.n, self.n)
        d = d + self.refine(torch.cat([d, x], 1))
        return x + d


class Model(nn.Module):
    """g + surrogate. Works in the instance-normalised frame (see module docstring)."""

    def __init__(self, d, n, K=8, identity=False, g_hidden=128, **kw):
        super().__init__()
        self.g = ParamEmbed(d, K, g_hidden, identity=identity)
        self.net = Surrogate(n=n, c_dim=self.g.K, **kw)

    def conditioning_parameters(self):
        return list(self.g.parameters()) + self.net.conditioning_parameters()

    def forward(self, x, theta, logr):
        return self.net(x, self.g(theta), logr)

    @torch.no_grad()
    def rescale_gauge(self, a):
        """c -> a c with the first conditioning layer absorbing 1/a: predictions are unchanged
        exactly, but the pullback metric of g scales by a^2 (the gauge of section 3.3)."""
        if self.g.identity:
            return
        last = self.g.net[-1]
        last.weight.mul_(a); last.bias.mul_(a)
        self.net.cond[0].weight[:, :self.g.K].div_(a)

    def forward_c(self, x, c, logr):
        return self.net(x, c, logr)


def normalise(s, eps=1e-8):
    """s: (B, f, n, n) -> (x = s / r, log r) with r the per-field RMS."""
    r = s.pow(2).mean((-2, -1)).sqrt().clamp_min(eps)
    return s / r[..., None, None], r.log()


def n_params(m):
    return sum(p.numel() for p in m.parameters())


In [ ]:
%%writefile src/unified_sbi/surrogate/fisher.py
"""Fisher machinery of docs/plasma_surrogate_conditioning.md, section 3.

All Jacobians are with respect to *standardised* theta and are measured in the
instance-normalised output frame y = s_{t+1} / r_t, so the output metric W is the identity in
that frame (a relative-error metric, see section 3.1).

    surrogate_jacobian   central differences of the surrogate in theta    (section 4.1)
    fisher_from_jacobian A = J^T J / D, pooled over a run's states          (section 3.2)
    FisherBank           per-run EMA of A, normalised target M = A / max(s, eps_s) + lam I
    metric_of_g          G_eps = J_g^T J_g + eps_g I for the embedding (differentiable)
    r_iso                tr P - log det P - d,  P = G_eps^{-1} M              (section 3.3)
    whitened_noise       delta ~ N(0, sigma^2 G^{-1})                        (section 3.5)
    maximin              greedy maximin selection in an embedding            (section 7)
"""

from __future__ import annotations

import torch
from torch.func import jacrev, vmap


def perturbations(theta, eps):
    """(S, d) -> (S, 2d, d): theta + eps e_i, theta - eps e_i for i = 1..d."""
    S, d = theta.shape
    E = torch.eye(d, device=theta.device, dtype=theta.dtype) * eps
    P = torch.stack([theta[:, None] + E, theta[:, None] - E], 2)    # (S, d, 2, d)
    return P.reshape(S, 2 * d, d)


def surrogate_jacobian(fn, x, theta, logr, eps=0.02, chunk=256):
    """J (S, d, f, n, n) of fn(x, theta, logr) by central differences in standardised theta.

    Gradients flow if called outside ``torch.no_grad`` (used for Jacobian supervision)."""
    S, d = theta.shape
    P = perturbations(theta, eps).reshape(-1, d)
    X = x.repeat_interleave(2 * d, 0)
    L = logr.repeat_interleave(2 * d, 0)
    outs = [fn(X[i:i + chunk], P[i:i + chunk], L[i:i + chunk]) for i in range(0, len(P), chunk)]
    Y = torch.cat(outs).reshape(S, d, 2, *x.shape[1:])
    return (Y[:, :, 0] - Y[:, :, 1]) / (2 * eps)


def fisher_from_jacobian(J):
    """(S, d, f, n, n) -> (S, d, d) with A = J^T J / D (D = f n n)."""
    Jf = J.flatten(2)
    return Jf @ Jf.transpose(1, 2) / Jf.shape[-1]


class FisherBank:
    """Per-run exponential moving average of the surrogate's Fisher matrix."""

    def __init__(self, n_runs, d, decay=0.7, device="cpu"):
        self.A = torch.zeros(n_runs, d, d, device=device)
        self.seen = torch.zeros(n_runs, dtype=torch.bool, device=device)
        self.decay, self.d = decay, d

    def update(self, run_ids, A):
        """run_ids: (S,) run index per sample; A: (S, d, d). Samples of one run are averaged."""
        run_ids = run_ids.to(self.A.device)
        u = torch.unique(run_ids)
        for r in u.tolist():
            a = A[run_ids == r].mean(0)
            if self.seen[r]:
                self.A[r] = self.decay * self.A[r] + (1 - self.decay) * a
            else:
                self.A[r], self.seen[r] = a, True

    def set(self, A):
        self.A = A.to(self.A.device).clone()
        self.seen[:] = True

    def scale(self):
        """s_bar = mean_r tr A_r / d over the runs seen so far."""
        return self.A[self.seen].diagonal(dim1=-2, dim2=-1).sum(-1).mean() / self.d

    def target(self, lam, eps_s=1e-12):
        """M = A / max(s_bar, eps_s) + lam I (section 3.2)."""
        eye = torch.eye(self.d, device=self.A.device)
        return self.A / self.scale().clamp_min(eps_s) + lam * eye


def jacobian_of_g(g, theta):
    """(N, d) -> J_g (N, K, d), differentiable w.r.t. the parameters of g."""
    return vmap(jacrev(lambda t: g(t[None])[0]))(theta)


def metric_of_g(g, theta, eps_g=0.0):
    """G_eps = J_g^T J_g + eps_g I  (N, d, d)."""
    Jg = jacobian_of_g(g, theta)
    G = Jg.transpose(1, 2) @ Jg
    return G + eps_g * torch.eye(G.shape[-1], device=G.device, dtype=G.dtype) if eps_g else G


def r_iso(G, M):
    """Mean over runs of tr P - log det P - d, P = G^{-1} M; zero iff G = M, >= 0.

    Computed as tr(G^{-1} M) + log det G - log det M - d with Cholesky solves (no explicit
    inverse). Pass G_eps = J_g^T J_g + eps_g I (``metric_of_g(..., eps_g)``) so the loss stays
    defined if J_g is temporarily rank-deficient."""
    d = G.shape[-1]
    L = torch.linalg.cholesky(G)
    tr = torch.cholesky_solve(M, L).diagonal(dim1=-2, dim2=-1).sum(-1)
    logdet_G = 2 * torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(-1)
    logdet_M = torch.linalg.slogdet(M)[1]
    return (tr - (logdet_M - logdet_G) - d).mean()


def whitened_noise(G, sigma, generator=None):
    """delta ~ N(0, sigma^2 G^{-1}) per row. G: (B, d, d) positive definite (use G_eps)."""
    B, d, _ = G.shape
    L = torch.linalg.cholesky(G)
    xi = torch.randn(B, d, 1, device=G.device, dtype=G.dtype, generator=generator)
    delta = torch.linalg.solve_triangular(L.transpose(1, 2), xi, upper=True)[..., 0]
    return torch.as_tensor(sigma, device=G.device, dtype=G.dtype).reshape(-1, 1) * delta


def isotropic_noise_matched(G, sigma, generator=None):
    """delta ~ N(0, s^2 I) with s^2 = sigma^2 d / tr G, so E[delta^T G delta] matches the
    whitened noise (the control arm A5)."""
    B, d, _ = G.shape
    s = torch.as_tensor(sigma, device=G.device, dtype=G.dtype).reshape(-1) * torch.sqrt(
        d / G.diagonal(dim1=-2, dim2=-1).sum(-1).clamp_min(1e-12))
    xi = torch.randn(B, d, device=G.device, dtype=G.dtype, generator=generator)
    return s[:, None] * xi


def nn_spacing(c):
    """Median nearest-neighbour distance between rows of c (N, K)."""
    D = torch.cdist(c, c)
    D.fill_diagonal_(float("inf"))
    return D.min(1).values.median()


def maximin(cand, existing, k):
    """Greedy maximin: pick k rows of ``cand`` (M, K) maximising the min Euclidean (chord)
    distance to ``existing`` (E, K) and to the rows already picked. Indices into cand."""
    dist = lambda i: (cand - cand[i]).norm(dim=1)
    d0 = torch.cdist(cand, existing).min(1).values if len(existing) else None
    return _greedy(len(cand), d0, dist, k)


def maximin_geodesic(cand, existing, k, n_neighbours=10):
    """Greedy maximin on graph shortest-path distances along the learned manifold (section 7):
    a symmetric k-nearest-neighbour graph over cand + existing in the embedding, weighted by
    chord length; geodesic ~ shortest path. Falls back to the chord for disconnected pairs."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import dijkstra
    X = torch.cat([cand, existing]).double().cpu()
    D = torch.cdist(X, X)
    nn = D.topk(n_neighbours + 1, largest=False).indices[:, 1:]
    rows = torch.arange(len(X))[:, None].expand_as(nn).reshape(-1)
    W = csr_matrix((D[rows, nn.reshape(-1)].numpy(), (rows.numpy(), nn.reshape(-1).numpy())),
                   shape=(len(X), len(X)))
    W = W.maximum(W.T)
    Dg = torch.as_tensor(dijkstra(W, directed=False))
    Dg = torch.where(torch.isfinite(Dg), Dg, D)
    M = len(cand)
    d0 = Dg[:M, M:].min(1).values if len(existing) else None
    return _greedy(M, d0, lambda i: Dg[:M, i], k)


def _greedy(M, d0, dist, k):
    dmin = d0.clone() if d0 is not None else torch.full((M,), float("inf"))
    dmin = dmin.double().cpu()
    picked = []
    for _ in range(k):
        i = int(torch.argmax(dmin))
        picked.append(i)
        dmin = torch.minimum(dmin, dist(i).double().cpu())
        dmin[i] = -1.0
    return picked


def affine_invariant_distance(A, B, jitter=1e-10):
    """|| log(A^{-1/2} B A^{-1/2}) ||_F per pair, for SPD (N, d, d)."""
    d = A.shape[-1]
    eye = torch.eye(d, dtype=A.dtype, device=A.device)
    ev = torch.linalg.eigvals(torch.linalg.solve(A + jitter * eye, B + jitter * eye)).real
    return torch.log(ev.clamp_min(1e-30)).pow(2).sum(-1).sqrt()


In [ ]:
%%writefile src/unified_sbi/surrogate/study.py
"""Few-run conditional surrogate study on the Hasegawa-Wakatani testbed.

Implements the ablation of docs/plasma_surrogate_conditioning.md, section 8:

    arm             conditioning                                         section
    A0_film         FiLM/adaLN on raw standardised theta (baseline)
    A1_g            learned g(theta), weight decay on the conditioning path
    A2_iso          A1 + R_iso against the surrogate's own Fisher A_self  3.2-3.3
    A3_iso_wn       A2 + Fisher-whitened theta-noise                     3.5
    A4_iso_jac      A2 + Jacobian supervision from branch restarts, with
                    the physical Fisher A_phys as the R_iso target        4.3
    A5_iso_in       A2 + isotropic theta-noise of matched magnitude       control for 3.5
    A6_full         A3 + Jacobian supervision + A_phys target             full method

The first decisive comparison is A0 / A1 / A2 / A4 (section 8); A3 vs A5 isolates the
whitening, and A6 is the full method.

One *cell* = (seed, budget N_theta, arm). Within a seed, budgets are nested prefixes of one
permutation of the simulated pool; every arm of a cell sees the same runs, the same
initialisation seed and the same number of optimiser steps. Validation runs (for picking the
best checkpoint) and test runs are disjoint from the pool and fixed across seeds and budgets:
all splits are *by run*.

Metrics (all on held-out test runs, in the relative frame y = s_{t+1} / rms(s_t)):
    one-step NRMSE per field, and persistence for reference;
    rollout NRMSE at several horizons and the mean radial flux Gamma_n of rollouts against
    the run's true mean flux (|log ratio| and Spearman rank correlation across test runs);
    sensitivity error against branch-restart Jacobians: relative Jacobian error (all and per
    parameter), cosine per parameter, Fisher log-det ratio and affine-invariant distance.

The design experiment (section 7, prediction P5) trains the design arm on N/2 random runs,
then adds N/2 runs chosen at random, by Latin hypercube, by maximin in standardised theta, by
maximin on the chord distance in the learned g-space, or by maximin on graph-geodesic distance
along the learned conditioning manifold; it simulates them and retrains on all N.
"""

from __future__ import annotations

import copy
import hashlib
import json
import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch

from . import fisher as F
from .hw import (HWConfig, branch_jacobian, flux, latin_hypercube, sample_prior, simulate,
                 standardise)
from .nets import Model, n_params, normalise

ARMS = {
    "A0_film":    dict(identity=True,  R=False, noise=None,       jac=False),
    "A1_g":       dict(identity=False, R=False, noise=None,       jac=False),
    "A2_iso":     dict(identity=False, R=True,  noise=None,       jac=False),
    "A3_iso_wn":  dict(identity=False, R=True,  noise="whitened", jac=False),
    "A4_iso_jac": dict(identity=False, R=True,  noise=None,       jac=True),
    "A5_iso_in":  dict(identity=False, R=True,  noise="iso",      jac=False),
    "A6_full":    dict(identity=False, R=True,  noise="whitened", jac=True),
}
DECISIVE = ("A0_film", "A1_g", "A2_iso", "A4_iso_jac")
DESIGNS = ("random", "lhs", "maximin_theta", "maximin_g", "maximin_geo")


@dataclass
class StudyConfig:
    hw: dict = field(default_factory=lambda: HWConfig().to_dict())
    # data (all splits by run)
    pool_size: int = 256
    n_val: int = 16
    n_test: int = 48
    data_seed: int = 0
    n_checkpoints: int = 4          # branch-restart states per run (pool and test)
    branch_eps: float = 0.02        # standardised units, simulator and surrogate alike
    budgets: tuple = (25, 50, 100, 200)
    seeds: tuple = (0, 1)
    arms: tuple = tuple(ARMS)
    # model
    K: int = 8
    g_hidden: int = 128
    width: int = 192
    depth: int = 6
    heads: int = 6
    patch: int = 4
    # optimisation
    steps: int = 8000
    batch: int = 32
    lr: float = 3e-4
    lr_warmup: int = 300
    wd: float = 1e-4
    wd_cond: float = 1e-2           # weight decay on g and every conditioning projection
    grad_clip: float = 1.0
    eval_every: int = 500
    n_val_pairs: int = 512
    # R_iso (section 3.3)
    r_start: float = 0.3            # fraction of steps with beta = 0 (warm-up, 4.2)
    r_ramp: float = 0.1             # fraction of steps to ramp beta up
    beta: float = 0.1
    lam: tuple = (0.3, 0.03)        # Fisher floor: held at lam[0], then linear to lam[1]
    lam_hold: float = 0.5           # fraction of the R phase before lam decays (Fisher estimate stable)
    eps_g: float = 1e-3             # G_eps = J_g^T J_g + eps_g I (section 3.3)
    eps_s: float = 1e-12            # s_eps = max(s_bar, eps_s) (section 3.2)
    fisher_every: int = 25
    fisher_runs: int = 16           # runs refreshed per update
    fisher_states: int = 2          # states per run per update
    fisher_decay: float = 0.7
    # Fisher-whitened smoothing (3.5): sigma_c = kappa * median NN spacing in g-space
    kappa_noise: tuple = (0.3, 0.1)
    # Jacobian supervision (4.3)
    gamma_jac: float = 0.1
    jac_batch: int = 4
    # evaluation
    rollout_starts: int = 3
    rollout_h: int = 40
    flux_from: int = 10             # rollout steps >= this enter the flux average
    horizons: tuple = (1, 2, 5, 10, 20, 40)
    # design experiment (section 7)
    design_budgets: tuple = (50,)
    design_arm: str = "A3_iso_wn"
    designs: tuple = DESIGNS
    design_neighbours: int = 10     # k of the kNN graph for geodesic maximin
    design_candidates: int = 2048
    # hardware
    device: str = "auto"
    sim_device: str = "auto"
    threads: int = 0

    def hcfg(self):
        return HWConfig(**{k: (tuple(v) if isinstance(v, list) else v) for k, v in self.hw.items()})

    def to_dict(self):
        return asdict(self)

    def data_key(self):
        h = {k: v for k, v in self.hw.items() if k != "batch"}
        keys = dict(hw=h, pool=self.pool_size, val=self.n_val, test=self.n_test,
                    seed=self.data_seed, ck=self.n_checkpoints, eps=self.branch_eps)
        return hashlib.sha1(json.dumps(keys, sort_keys=True, default=list).encode()).hexdigest()[:10]


def pick_device(name):
    if name == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return name


# ---------------------------------------------------------------------------------------------
# data


class HWData:
    """Simulated runs (pool + validation + test), cached to disk.

    Run index layout: [0, pool) pool, [pool, pool + val) validation, then test, then any
    runs added by the design experiment."""

    def __init__(self, cfg: StudyConfig, cache, log=print):
        self.cfg, self.h, self.log = cfg, cfg.hcfg(), log
        self.cache = Path(cache)
        self.cache.mkdir(parents=True, exist_ok=True)
        path = self.cache / f"hw_{cfg.data_key()}.pt"
        if path.exists():
            d = torch.load(path, weights_only=False)
            log(f"loaded {path}")
        else:
            d = self._generate()
            torch.save(d, path)
            log(f"saved {path}")
        self.theta = d["theta"]                   # (R, d) natural, float64
        self.states = d["states"]                 # (R, T, 2, n, n) float32
        self.flux = d["flux"]                     # (R, T)
        self.finite = d["finite"]
        self.ck = d["ck"]                         # (R, n_ck) checkpoint time indices
        self.J = d["J"]                           # (R, n_ck, d, 2, n, n) relative frame (nan if absent)
        self.theta_std = standardise(self.theta, self.h).float()
        P, V, T = cfg.pool_size, cfg.n_val, cfg.n_test
        fin = self.finite.numpy()
        self.pool = [i for i in range(P) if fin[i]]
        self.val = [i for i in range(P, P + V) if fin[i]]
        self.test = [i for i in range(P + V, P + V + T) if fin[i]]
        bad = int((~self.finite).sum())
        if bad:
            log(f"warning: {bad} runs were not finite and are excluded")

    def _simulate(self, theta, seeds, with_branches):
        h, dev = self.h, pick_device(self.cfg.sim_device)
        t0 = time.time()
        r = simulate(theta, seeds, h, device=dev, log=self.log)
        n_ck = self.cfg.n_checkpoints
        T = h.n_save
        ck = torch.from_numpy(np.linspace(0, T - 2, n_ck).round().astype(np.int64))[None].repeat(len(theta), 1)
        J = torch.full((len(theta), n_ck, h.d, 2, h.n, h.n), float("nan"))
        if with_branches.any():
            idx = np.where(with_branches)[0]
            S = r["states"][torch.as_tensor(idx)][:, ck[0]].reshape(-1, 2, h.n, h.n)
            th = np.repeat(np.asarray(theta)[idx], n_ck, 0)
            Jb = torch.empty(len(S), h.d, 2, h.n, h.n)
            for i in range(0, len(S), 128):                               # bounded memory
                Jb[i:i + 128], _ = branch_jacobian(S[i:i + 128], th[i:i + 128], h,
                                                   eps=self.cfg.branch_eps, device=dev)
            rms = S.pow(2).mean((-2, -1)).sqrt()
            Jb = Jb / rms[:, None, :, None, None]
            J[torch.as_tensor(idx)] = Jb.reshape(len(idx), n_ck, *Jb.shape[1:])
            self.log(f"branch restarts for {len(idx)} runs x {n_ck} checkpoints done")
        self.log(f"simulated {len(theta)} runs in {time.time() - t0:.0f} s")
        return dict(theta=torch.as_tensor(np.asarray(theta)), states=r["states"], flux=r["flux"],
                    finite=r["finite"], ck=ck, J=J)

    def _generate(self):
        c, h = self.cfg, self.h
        rng = np.random.default_rng(c.data_seed)
        n = c.pool_size + c.n_val + c.n_test
        theta = sample_prior(n, h, rng)
        seeds = c.data_seed * 1_000_000 + np.arange(n)
        branches = np.ones(n, bool)
        branches[c.pool_size:c.pool_size + c.n_val] = False             # validation needs none
        self.log(f"simulating {n} runs ({h.n}^2, {h.n_save} snapshots, spin-up {h.t_spin})")
        return self._simulate(theta, seeds, branches)

    def add_runs(self, theta, tag, with_branches=False):
        """Simulate extra runs (design experiment), cache them, append, return their indices."""
        path = self.cache / f"hw_{self.cfg.data_key()}_extra_{tag}.pt"
        if path.exists():
            d = torch.load(path, weights_only=False)
        else:
            off = int(hashlib.sha1(tag.encode()).hexdigest()[:6], 16)
            seeds = 7_000_000_000 + off * 10_000 + np.arange(len(theta))
            d = self._simulate(theta, seeds, np.full(len(theta), with_branches))
            torch.save(d, path)
        start = len(self.theta)
        self.theta = torch.cat([self.theta, d["theta"]])
        self.states = torch.cat([self.states, d["states"]])
        self.flux = torch.cat([self.flux, d["flux"]])
        self.finite = torch.cat([self.finite, d["finite"]])
        self.ck = torch.cat([self.ck, d["ck"]])
        self.J = torch.cat([self.J, d["J"]])
        self.theta_std = standardise(self.theta, self.h).float()
        return [start + i for i in range(len(d["theta"])) if bool(d["finite"][i])]

    def pool_order(self, seed):
        rng = np.random.default_rng(1000 + seed)
        return [self.pool[i] for i in rng.permutation(len(self.pool))]

    def pairs(self, runs, t):
        """Normalised (x, logr, y) for runs[k] at time t[k] -> t[k] + 1."""
        s0 = self.states[runs, t]
        s1 = self.states[runs, t + 1]
        x, logr = normalise(s0)
        y = s1 / logr.exp()[..., None, None]
        return x, logr, y


# ---------------------------------------------------------------------------------------------
# training


def build_model(arm, cfg: StudyConfig, data: HWData):
    a = ARMS[arm]
    return Model(data.h.d, data.h.n, K=cfg.K, identity=a["identity"], g_hidden=cfg.g_hidden,
                 width=cfg.width, depth=cfg.depth, heads=cfg.heads, patch=cfg.patch)


def _lr_factor(step, cfg):
    if step < cfg.lr_warmup:
        return (step + 1) / cfg.lr_warmup
    p = (step - cfg.lr_warmup) / max(1, cfg.steps - cfg.lr_warmup)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * p))


def _val_pairs(cfg, data, seed=0):
    rng = np.random.default_rng(5000 + seed)
    T = data.states.shape[1]
    runs = rng.choice(data.val, cfg.n_val_pairs)
    t = rng.integers(0, T - 1, cfg.n_val_pairs)
    return torch.as_tensor(runs), torch.as_tensor(t)


@torch.no_grad()
def one_step_mse(model, data, runs, t, dev, bs=256):
    tot, n = 0.0, 0
    for i in range(0, len(runs), bs):
        r, tt = runs[i:i + bs], t[i:i + bs]
        x, logr, y = (v.to(dev) for v in data.pairs(r, tt))
        pred = model(x, data.theta_std[r].to(dev), logr)
        tot += (pred - y).pow(2).mean((1, 2, 3)).sum().item()
        n += len(r)
    return tot / n


def _exact_float32():
    """Finite-difference Jacobians (eps = 0.02) need full fp32: TF32 convolutions/matmuls
    (cuDNN's default on Ampere) add ~1e-3 relative rounding noise, i.e. tens of per cent of J."""
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


def train(arm, cfg: StudyConfig, data: HWData, train_runs, seed, log=print):
    a = ARMS[arm]
    dev = pick_device(cfg.device)
    _exact_float32()
    torch.manual_seed(10_000 * seed + len(train_runs))
    model = build_model(arm, cfg, data).to(dev)
    cond = {id(p) for p in model.conditioning_parameters()}
    main = [p for p in model.parameters() if id(p) not in cond]
    opt = torch.optim.AdamW([dict(params=main, weight_decay=cfg.wd),
                             dict(params=model.conditioning_parameters(), weight_decay=cfg.wd_cond)],
                            lr=cfg.lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: _lr_factor(s, cfg))
    gen = torch.Generator().manual_seed(seed)
    ngen = torch.Generator(device=dev).manual_seed(seed + 1)
    runs_t = torch.as_tensor(train_runs)
    N, d = len(train_runs), data.h.d
    T = data.states.shape[1]
    th_train = data.theta_std[runs_t].to(dev)
    vr, vt = _val_pairs(cfg, data)

    bank = F.FisherBank(N, d, cfg.fisher_decay, dev)
    if a["jac"]:                                    # physical Fisher A_phys from branch restarts (4.3)
        bank.set(F.fisher_from_jacobian(data.J[runs_t].flatten(0, 1)).reshape(N, -1, d, d).mean(1))
        ck_states = data.states[runs_t[:, None], data.ck[runs_t]]          # (N, n_ck, 2, n, n)
        ck_J = data.J[runs_t]
    r_on = int(cfg.r_start * cfg.steps) if a["R"] else cfg.steps + 1
    ramp = max(1, int(cfg.r_ramp * cfg.steps))
    G_cache, sigma_c = None, 0.0

    def refresh_bank(local):
        """Surrogate Fisher (section 4.1) for the given local run indices."""
        loc = torch.as_tensor(local).repeat_interleave(cfg.fisher_states)
        t = torch.randint(0, T - 1, (len(loc),), generator=gen)
        x, logr, _ = data.pairs(runs_t[loc], t)
        with torch.no_grad():
            J = F.surrogate_jacobian(model, x.to(dev), th_train[loc.to(dev)], logr.to(dev),
                                     cfg.branch_eps)
        bank.update(loc.to(dev), F.fisher_from_jacobian(J))

    hist, best, t_start = [], (float("inf"), None, -1), time.time()
    for step in range(cfg.steps):
        model.train()
        phase = min(1.0, max(0.0, (step - r_on) / max(1, cfg.steps - r_on)))
        lp = min(1.0, max(0.0, (phase - cfg.lam_hold) / max(1e-9, 1 - cfg.lam_hold)))
        lam = cfg.lam[0] + (cfg.lam[1] - cfg.lam[0]) * lp
        kap = cfg.kappa_noise[0] + (cfg.kappa_noise[1] - cfg.kappa_noise[0]) * phase
        beta = cfg.beta * min(1.0, (step - r_on) / ramp) if step >= r_on else 0.0
        if a["R"] and step >= r_on and (step == r_on or step % cfg.fisher_every == 0):
            if not a["jac"]:
                if step == r_on:
                    for i in range(0, N, 64):
                        refresh_bank(list(range(i, min(N, i + 64))))
                else:
                    refresh_bank(torch.randperm(N, generator=gen)[:cfg.fisher_runs].tolist())
            if step == r_on:
                # gauge fix before R_iso switches on: rescale c -> a c (predictions unchanged)
                # with a^2 = mean_r tr(G_r^{-1} M_r) / d, the optimal scalar for R_iso
                with torch.no_grad():
                    G0 = F.metric_of_g(model.g, th_train)
                    M0 = bank.target(lam, cfg.eps_s)
                    L0 = torch.linalg.cholesky(G0 + cfg.eps_g * torch.eye(d, device=dev))
                    a2 = torch.cholesky_solve(M0, L0).diagonal(dim1=-2, dim2=-1).sum(-1).mean() / d
                model.rescale_gauge(float(a2.sqrt()))
                log(f"    [{arm}] R_iso on at step {step}: gauge rescale a = {float(a2.sqrt()):.3g}")
            with torch.no_grad():
                G_cache = F.metric_of_g(model.g, th_train, cfg.eps_g)
                sigma_c = kap * F.nn_spacing(model.g(th_train)).item() if N > 1 else 0.0

        loc = torch.randint(0, N, (cfg.batch,), generator=gen)
        t = torch.randint(0, T - 1, (cfg.batch,), generator=gen)
        x, logr, y = (v.to(dev) for v in data.pairs(runs_t[loc], t))
        th = th_train[loc.to(dev)]
        if a["noise"] and step >= r_on and G_cache is not None:
            Gb = G_cache[loc.to(dev)]
            noise = F.whitened_noise if a["noise"] == "whitened" else F.isotropic_noise_matched
            th = th + noise(Gb, sigma_c, ngen)
        pred = model(x, th, logr)
        l_pred = (pred - y).pow(2).mean()
        loss = l_pred
        l_r = l_j = torch.zeros((), device=dev)
        if beta > 0:
            G = F.metric_of_g(model.g, th_train, cfg.eps_g)
            l_r = F.r_iso(G, bank.target(lam, cfg.eps_s))
            loss = loss + beta * l_r
        if a["jac"]:
            k = torch.randint(0, N * cfg.n_checkpoints, (cfg.jac_batch,), generator=gen)
            rl, cl = k // cfg.n_checkpoints, k % cfg.n_checkpoints
            xs, lr_ = normalise(ck_states[rl, cl].to(dev))
            Jt = ck_J[rl, cl].to(dev)
            Jh = F.surrogate_jacobian(model, xs, th_train[rl.to(dev)], lr_, cfg.branch_eps)
            l_j = (Jh - Jt).pow(2).mean() / Jt.pow(2).mean().clamp_min(1e-12)
            loss = loss + cfg.gamma_jac * l_j
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step(); sched.step()
        if (step + 1) % cfg.eval_every == 0 or step + 1 == cfg.steps:
            model.eval()
            v = one_step_mse(model, data, vr, vt, dev)
            hist.append(dict(step=step + 1, train=l_pred.item(), val=v, r_iso=l_r.item(),
                             jac=l_j.item(), beta=beta, sigma_c=sigma_c))
            log(f"    [{arm} N={N} s={seed}] step {step + 1:5d}  train {l_pred.item():.4f}  "
                f"val {v:.4f}  R {l_r.item():.3f}  jac {l_j.item():.3f}  "
                f"({time.time() - t_start:.0f} s)")
            if v < best[0]:
                best = (v, copy.deepcopy({k: w.detach().cpu() for k, w in model.state_dict().items()}),
                        step + 1)
    model.load_state_dict(best[1])
    model.eval()
    info = dict(best_val=best[0], best_step=best[2], train_time=time.time() - t_start, history=hist,
                r_iso_final=hist[-1]["r_iso"] if hist else float("nan"))
    return model, info


# ---------------------------------------------------------------------------------------------
# evaluation


def _spearman(a, b):
    a, b = np.asarray(a), np.asarray(b)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return float("nan")
    ra = np.argsort(np.argsort(a[ok])); rb = np.argsort(np.argsort(b[ok]))
    return float(np.corrcoef(ra, rb)[0, 1])


@torch.no_grad()
def evaluate(model, cfg: StudyConfig, data: HWData, train_runs, runs=None, bs=128):
    dev = pick_device(cfg.device)
    _exact_float32()
    model.eval()
    runs = data.test if runs is None else runs
    T = data.states.shape[1]
    d = data.h.d
    k0_idx = 4
    out = {}

    # 1. one-step errors on every transition of every test run
    os_err, pers = [], []
    for r in runs:
        rr = torch.full((T - 1,), r)
        tt = torch.arange(T - 1)
        e = torch.zeros(2); p = torch.zeros(2)
        for i in range(0, T - 1, bs):
            x, logr, y = (v.to(dev) for v in data.pairs(rr[i:i + bs], tt[i:i + bs]))
            pred = model(x, data.theta_std[rr[i:i + bs]].to(dev), logr)
            e += (pred - y).pow(2).mean((-2, -1)).sum(0).cpu()
            p += (x - y).pow(2).mean((-2, -1)).sum(0).cpu()
        os_err.append((e / (T - 1)).sqrt().numpy()); pers.append((p / (T - 1)).sqrt().numpy())
    os_err, pers = np.array(os_err), np.array(pers)
    out["onestep_nrmse_n"], out["onestep_nrmse_phi"] = map(float, np.median(os_err, 0))
    out["onestep_nrmse"] = float(np.median(os_err.mean(1)))
    out["persistence_nrmse"] = float(np.median(pers.mean(1)))

    # 2. rollouts
    H = min(cfg.rollout_h, T - 1)
    starts = np.linspace(0, T - 1 - H, cfg.rollout_starts).round().astype(int)
    hz = [h for h in cfg.horizons if h <= H]
    roll = np.full((len(runs), len(starts), len(hz)), np.nan)
    fl_pred = np.full((len(runs), len(starts)), np.nan)
    blow = 0
    for i, r in enumerate(runs):
        th = data.theta_std[[r] * len(starts)].to(dev)
        k0 = data.theta[r, k0_idx].item()
        s = data.states[r, starts].to(dev)                          # (S, 2, n, n)
        fl = []
        for k in range(1, H + 1):
            x, logr = normalise(s)
            s = model(x, th, logr) * logr.exp()[..., None, None]
            if k >= cfg.flux_from:
                fl.append(flux(s, k0).cpu())
            if k in hz:
                truth = data.states[r, starts + k].to(dev)
                err = ((s - truth).pow(2).mean((-3, -2, -1)) / truth.pow(2).mean((-3, -2, -1))).sqrt()
                roll[i, :, hz.index(k)] = err.cpu().numpy()
        bad = ~torch.isfinite(s).flatten(1).all(1).cpu()
        blow += int(bad.sum())
        if fl:
            f = torch.stack(fl).mean(0).numpy()
            f[bad.numpy()] = np.nan
            fl_pred[i] = f
    true_flux = data.flux[runs].mean(1).numpy()
    with np.errstate(all="ignore"):
        pred_flux = np.nanmean(fl_pred, 1)
    for j, h in enumerate(hz):
        out[f"rollout_nrmse_h{h}"] = float(np.nanmedian(roll[:, :, j]))
    out["rollout_blowups"] = blow
    # flux spans ~3 decades over the prior: relative error (a sign error counts as >= 1) and
    # the rank correlation across test runs (does the rollout order the runs' transport?)
    out["flux_rel_err"] = float(np.nanmedian(np.abs(pred_flux / true_flux - 1)))
    out["flux_spearman"] = _spearman(pred_flux, true_flux)

    # 3. sensitivity against branch-restart Jacobians
    have = [r for r in runs if torch.isfinite(data.J[r]).all()]
    if have:
        rj = torch.as_tensor(have)
        S = data.states[rj[:, None], data.ck[rj]].flatten(0, 1)           # (R*ck, 2, n, n)
        Jt = data.J[rj].flatten(0, 1)                                       # (R*ck, d, 2, n, n)
        th = data.theta_std[rj].repeat_interleave(cfg.n_checkpoints, 0)
        Jh = []
        for i in range(0, len(S), 16):
            x, logr = normalise(S[i:i + 16].to(dev))
            Jh.append(F.surrogate_jacobian(model, x, th[i:i + 16].to(dev), logr, cfg.branch_eps).cpu())
        Jh = torch.cat(Jh)
        num = (Jh - Jt).flatten(1).norm(dim=1); den = Jt.flatten(1).norm(dim=1)
        out["jac_rel_err"] = float((num / den).median())
        pp = (Jh - Jt).flatten(2).norm(dim=2) / Jt.flatten(2).norm(dim=2)        # (S, d)
        cos = torch.nn.functional.cosine_similarity(Jh.flatten(2), Jt.flatten(2), dim=2)
        out["jac_rel_err_per_param"] = pp.median(0).values.tolist()
        out["jac_cos_per_param"] = cos.mean(0).tolist()
        At = F.fisher_from_jacobian(Jt).reshape(len(have), -1, d, d).mean(1).double()
        Ah = F.fisher_from_jacobian(Jh).reshape(len(have), -1, d, d).mean(1).double()
        # regularised log-det ratio: same scale s (from the true Fisher) and same floor in both
        s_t = At.diagonal(dim1=-2, dim2=-1).sum(-1).mean() / d
        eye = torch.eye(d, dtype=At.dtype)
        Mh, Mt = Ah / s_t + cfg.lam[1] * eye, At / s_t + cfg.lam[1] * eye
        ld = torch.linalg.slogdet(Mh)[1] - torch.linalg.slogdet(Mt)[1]
        out["fisher_logdet_ratio"] = float(ld.median())
        tn = lambda A: A / A.diagonal(dim1=-2, dim2=-1).sum(-1)[:, None, None]
        out["fisher_shape_dist"] = float(F.affine_invariant_distance(tn(At), tn(Ah)).median())
        out["fisher_true_condition"] = float(torch.linalg.cond(At).median())

    # 4. global folding of g (section 10): chord |g(a) - g(b)| over the local Fisher length
    #    sqrt(dtheta^T G(mid) dtheta) for random prior pairs; values << 1 at large separation
    #    mean distant theta map close together in c.
    if not model.g.identity:
        gen = torch.Generator().manual_seed(123)
        ta = torch.rand(2048, d, generator=gen) * 2 * math.sqrt(3) - math.sqrt(3)
        tb = torch.rand(2048, d, generator=gen) * 2 * math.sqrt(3) - math.sqrt(3)
        with torch.enable_grad():
            Gm = F.metric_of_g(model.g, ((ta + tb) / 2).to(dev)).detach().cpu()
        dt = tb - ta
        loc = torch.einsum("bi,bij,bj->b", dt, Gm, dt).clamp_min(1e-12).sqrt()
        chord = (model.g(ta.to(dev)) - model.g(tb.to(dev))).norm(dim=1).cpu()
        far = dt.norm(dim=1) > dt.norm(dim=1).median()
        out["g_fold_ratio_p01"] = float(torch.quantile(chord[far] / loc[far], 0.01))

    # 5. per-run arrays: error vs distance to the nearest training run (standardised theta)
    tr = data.theta_std[train_runs]
    dist = torch.cdist(data.theta_std[runs], tr).min(1).values.numpy()
    arrays = dict(onestep=os_err.mean(1), dist=dist, flux_pred=pred_flux, flux_true=true_flux,
                  theta=data.theta[runs].numpy())
    return out, arrays


# ---------------------------------------------------------------------------------------------
# cells, studies and the design experiment


def _save(path: Path, metrics, arrays, model=None):
    path.mkdir(parents=True, exist_ok=True)
    np.savez(path / "arrays.npz", **arrays)
    if model is not None:
        torch.save(model.state_dict(), path / "model.pt")
    (path / "metrics.json").write_text(json.dumps(metrics, indent=1))


def run_cell(cfg: StudyConfig, data: HWData, seed, budget, arm, out: Path, log=print, save_model=True):
    path = out / f"seed{seed}" / f"N{budget}" / arm
    if (path / "metrics.json").exists():
        log(f"  skip {path} (done)")
        return json.loads((path / "metrics.json").read_text())
    runs = data.pool_order(seed)[:budget]
    if len(runs) < budget:
        log(f"  only {len(runs)} finite pool runs for budget {budget}")
    log(f"  cell seed={seed} N={budget} arm={arm}")
    model, info = train(arm, cfg, data, runs, seed, log)
    m, arrays = evaluate(model, cfg, data, runs)
    m.update(info, arm=arm, budget=budget, seed=seed, n_params=n_params(model),
             n_params_cond=sum(p.numel() for p in model.conditioning_parameters()))
    _save(path, m, arrays, model if save_model else None)
    log(f"  -> one-step {m['onestep_nrmse']:.4f}  rollout@10 {m.get('rollout_nrmse_h10', float('nan')):.3f}  "
        f"flux rel {m['flux_rel_err']:.3f}  jac {m.get('jac_rel_err', float('nan')):.3f}")
    return m


def _check_config(cfg: StudyConfig, out: Path):
    out.mkdir(parents=True, exist_ok=True)
    cpath = out / "config.json"
    free = {"arms", "seeds", "budgets", "design_budgets", "device", "sim_device", "threads"}
    mine = {k: v for k, v in json.loads(json.dumps(cfg.to_dict(), default=list)).items() if k not in free}
    if cpath.exists():
        old = {k: v for k, v in json.loads(cpath.read_text()).items() if k not in free}
        if old != mine:
            diff = sorted(k for k in set(old) | set(mine) if old.get(k) != mine.get(k))
            raise RuntimeError(f"{out} holds results for a different configuration ({diff}); "
                               "use a new output folder")
    cpath.write_text(json.dumps(json.loads(json.dumps(cfg.to_dict(), default=list)), indent=1))


def run_study(cfg: StudyConfig, out, cache, cells=None, log=print):
    if cfg.threads:
        torch.set_num_threads(cfg.threads)
    out = Path(out)
    _check_config(cfg, out)
    data = HWData(cfg, cache, log)
    cells = cells or [(s, n, a) for s in cfg.seeds for n in cfg.budgets for a in cfg.arms]
    for s, n, a in cells:
        run_cell(cfg, data, s, n, a, out, log)
    return data


def run_design(cfg: StudyConfig, out, cache, data=None, log=print):
    """Section 7 / prediction P5: random vs maximin-theta vs maximin-g additions."""
    if cfg.threads:
        torch.set_num_threads(cfg.threads)
    out = Path(out)
    _check_config(cfg, out)
    data = data or HWData(cfg, cache, log)
    arm, dev = cfg.design_arm, pick_device(cfg.device)
    for seed in cfg.seeds:
        for N in cfg.design_budgets:
            base_dir = out / "design" / f"seed{seed}" / f"N{N}"
            half = N // 2
            order = data.pool_order(seed)
            base = order[:half]
            todo = [g for g in cfg.designs if not (base_dir / g / "metrics.json").exists()]
            if not todo:
                log(f"  skip {base_dir} (done)")
                continue
            log(f"  design seed={seed} N={N}: stage model on {half} runs")
            stage, _ = train(arm, cfg, data, base, seed, log)
            rng = np.random.default_rng(9000 + seed)
            cand = sample_prior(cfg.design_candidates, data.h, rng)
            cand_std = standardise(torch.as_tensor(cand), data.h).float()
            with torch.no_grad():
                cg = stage.g(cand_std.to(dev)).cpu()
                bg = stage.g(data.theta_std[base].to(dev)).cpu()
            k = N - half
            picks = dict(
                lhs=lambda: latin_hypercube(k, data.h, np.random.default_rng(9100 + seed)),
                maximin_theta=lambda: cand[F.maximin(cand_std, data.theta_std[base], k)],
                maximin_g=lambda: cand[F.maximin(cg, bg, k)],
                maximin_geo=lambda: cand[F.maximin_geodesic(cg, bg, k, cfg.design_neighbours)])
            for g in todo:
                if g == "random":
                    added = order[half:N]
                else:
                    added = data.add_runs(picks[g](), f"s{seed}_N{N}_{g}",
                                          with_branches=ARMS[arm]["jac"])
                runs = base + added
                log(f"  design {g}: training {arm} on {len(runs)} runs")
                model, info = train(arm, cfg, data, runs, seed, log)
                m, arrays = evaluate(model, cfg, data, runs)
                m.update(info, arm=arm, budget=N, seed=seed, design=g,
                         added_theta=data.theta[added].tolist())
                _save(base_dir / g, m, arrays)
                log(f"  -> {g}: one-step {m['onestep_nrmse']:.4f}  jac {m.get('jac_rel_err', float('nan')):.3f}")
    return data


def load_results(out):
    rows = []
    for p in Path(out).glob("seed*/N*/*/metrics.json"):
        m = json.loads(p.read_text())
        rows.append({k: v for k, v in m.items() if not isinstance(v, (list, dict))})
    return rows


In [ ]:
%%writefile scripts/plot_surrogate.py
#!/usr/bin/env python3
"""Figures and tables for the few-run surrogate study.

    python scripts/plot_surrogate.py --results results/surrogate

Writes <results>/figures/{scaling,sensitivity_by_param,design}.png and
<results>/tables/{summary,paired_vs_A0,design}.csv.
"""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# reference categorical palette, fixed order (validated: CVD and normal-vision checks pass;
# contrast WARN on aqua/yellow/magenta -> every series also gets a marker + line style and
# the tables carry the numbers)
ARM_STYLE = {
    "A0_film":    ("#2a78d6", "--", "o", "A0 FiLM on θ"),
    "A1_g":       ("#eb6834", "--", "s", "A1 learned g"),
    "A2_iso":     ("#1baf7a", "-",  "D", "A2 + R_iso (self-Fisher)"),
    "A3_iso_wn":  ("#eda100", "-",  "^", "A3 A2 + whitened noise"),
    "A4_iso_jac": ("#e87ba4", "-",  "v", "A4 A2 + Jacobian sup. (phys-Fisher)"),
    "A5_iso_in":  ("#008300", ":",  "X", "A5 A2 + isotropic noise"),
    "A6_full":    ("#4a3aa7", "-",  "P", "A6 full method"),
}
DESIGN_COL = {"random": "#2a78d6", "lhs": "#eb6834", "maximin_theta": "#1baf7a",
              "maximin_g": "#eda100", "maximin_geo": "#e87ba4"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
PARAMS = ["log α", "κ", "log ν̂", "log D̂", "k0"]

PANELS = [
    ("onestep_nrmse", "one-step NRMSE (held-out runs)", True),
    ("rollout_nrmse_h10", "rollout NRMSE at 10 steps", True),
    ("flux_rel_err", "flux |Γ̂/Γ − 1| (rollouts)", True),
    ("flux_spearman", "flux rank corr. across runs ↑", False),
    ("jac_rel_err", "Jacobian rel. error vs branches", True),
    ("fisher_logdet_ratio", "regularised Fisher log-det ratio (0 = right)", False),
]


def style(ax):
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelsize=8)


def load(results):
    rows = []
    for p in Path(results).glob("seed*/N*/*/metrics.json"):
        m = json.loads(p.read_text())
        row = {k: v for k, v in m.items() if not isinstance(v, (list, dict))}
        for key in ("jac_rel_err_per_param", "jac_cos_per_param"):
            for i, v in enumerate(m.get(key, [])):
                row[f"{key}_{i}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


def scaling_figure(df, out):
    arms = [a for a in ARM_STYLE if a in set(df.arm)]
    fig, axes = plt.subplots(2, 3, figsize=(12, 6.6), constrained_layout=True)
    for ax, (key, title, logy) in zip(axes.flat, PANELS):
        if key not in df:
            ax.set_visible(False)
            continue
        for a in arms:
            g = df[df.arm == a].groupby("budget")[key]
            m, lo, hi = g.mean(), g.min(), g.max()
            c, ls, mk, lab = ARM_STYLE[a]
            ax.errorbar(m.index, m.values, yerr=[m - lo, hi - m], color=c, ls=ls, marker=mk,
                        ms=6, lw=2, capsize=0, elinewidth=1, label=lab)
        ax.set_xscale("log")
        if key == "fisher_logdet_ratio":
            ax.axhline(0, color=MUTED, lw=1)
        if logy:
            ax.set_yscale("log")
        ax.set_xticks(sorted(df.budget.unique()))
        ax.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
        ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
        ax.set_title(title, fontsize=10, color=INK, loc="left")
        ax.set_xlabel("training runs N_θ", fontsize=8, color=MUTED)
        style(ax)
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="outside lower center", ncol=min(4, len(arms)), frameon=False, fontsize=9)
    fig.suptitle("Held-out-θ accuracy and sensitivity vs number of runs (mean over seeds, bars = min–max)",
                 fontsize=11, color=INK, x=0.01, ha="left")
    fig.savefig(out / "scaling.png", dpi=150)
    plt.close(fig)


def sensitivity_figure(df, out):
    if "jac_rel_err_per_param_0" not in df:
        return
    Nmax = df.budget.max()
    arms = [a for a in ARM_STYLE if a in set(df.arm)]
    sub = df[df.budget == Nmax]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
    w = 0.8 / len(arms)
    x = np.arange(len(PARAMS))
    for ax, key, title in [(axes[0], "jac_rel_err_per_param", f"Jacobian rel. error per parameter (N_θ = {Nmax})"),
                           (axes[1], "jac_cos_per_param", f"cosine(Ĵ_i, J_i) per parameter ↑ (N_θ = {Nmax})")]:
        for j, a in enumerate(arms):
            vals = [sub[sub.arm == a][f"{key}_{i}"].mean() for i in range(len(PARAMS))]
            c, _, _, lab = ARM_STYLE[a]
            ax.bar(x + (j - (len(arms) - 1) / 2) * w, vals, w * 0.9, color=c, label=lab,
                   edgecolor="white", linewidth=1)
        ax.set_xticks(x, PARAMS)
        ax.set_title(title, fontsize=10, color=INK, loc="left")
        style(ax)
    axes[0].set_yscale("log")
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="outside lower center", ncol=min(4, len(arms)), frameon=False, fontsize=8)
    fig.savefig(out / "sensitivity_by_param.png", dpi=150)
    plt.close(fig)


def design_figure(results, out, tables):
    rows = []
    for p in Path(results).glob("design/seed*/N*/*/metrics.json"):
        m = json.loads(p.read_text())
        rows.append({k: v for k, v in m.items() if not isinstance(v, (list, dict))})
    if not rows:
        return
    df = pd.DataFrame(rows)
    df.groupby(["budget", "design"]).mean(numeric_only=True).to_csv(tables / "design.csv")
    keys = [("onestep_nrmse", "one-step NRMSE"), ("rollout_nrmse_h10", "rollout NRMSE @10"),
            ("jac_rel_err", "Jacobian rel. error")]
    budgets = sorted(df.budget.unique())
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
    for ax, (k, t) in zip(axes, keys):
        if k not in df or df[k].isna().all():
            ax.set_visible(False)
            continue
        x = np.arange(len(budgets))
        present = [g for g in DESIGN_COL if g in set(df.design)]
        w = 0.8 / len(present)
        for j, g in enumerate(present):
            s = df[df.design == g].groupby("budget")[k]
            m = s.mean().reindex(budgets)
            lo, hi = s.min().reindex(budgets), s.max().reindex(budgets)
            ax.bar(x + (j - (len(present) - 1) / 2) * w, m, w * 0.92, color=DESIGN_COL[g], label=g,
                   edgecolor="white",
                   yerr=[m - lo, hi - m], error_kw=dict(elinewidth=1, ecolor=MUTED))
        ax.set_xticks(x, [f"N_θ = {b}" for b in budgets])
        ax.set_title(t, fontsize=10, color=INK, loc="left")
        style(ax)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="outside lower center", ncol=5, frameon=False, fontsize=9)
    fig.savefig(out / "design.png", dpi=150)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", required=True)
    a = ap.parse_args()
    res = Path(a.results)
    figs, tables = res / "figures", res / "tables"
    figs.mkdir(exist_ok=True, parents=True); tables.mkdir(exist_ok=True)
    df = load(res)
    if len(df):
        df.to_csv(tables / "all_cells.csv", index=False)
        num = df.drop(columns=["seed"]).groupby(["arm", "budget"]).mean(numeric_only=True)
        num.to_csv(tables / "summary.csv")
        # paired seed-level differences against the FiLM baseline (positive = better than A0)
        keys = [k for k, _, _ in PANELS if k in df]
        # the log-det ratio's target is 0, so compare its magnitude
        absd = df.assign(**({"fisher_logdet_ratio": df.fisher_logdet_ratio.abs()}
                            if "fisher_logdet_ratio" in df else {}))
        base = absd[absd.arm == "A0_film"].set_index(["seed", "budget"])[keys]
        out = []
        for arm in sorted(set(df.arm) - {"A0_film"}):
            d = absd[absd.arm == arm].set_index(["seed", "budget"])[keys]
            diff = (base - d).dropna(how="all")
            if "flux_spearman" in diff:
                diff["flux_spearman"] *= -1
            g = diff.groupby("budget")
            t = g.mean().add_suffix("_gain").join(g.sem().add_suffix("_se"))
            t["arm"] = arm
            out.append(t.reset_index())
        if out:
            pd.concat(out).to_csv(tables / "paired_vs_A0.csv", index=False)
        scaling_figure(df, figs)
        sensitivity_figure(df, figs)
        with pd.option_context("display.width", 200, "display.max_columns", 20):
            cols = [k for k, _, _ in PANELS if k in num]
            print(num[cols].round(4))
    design_figure(res, figs, tables)
    print("figures in", figs)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_surrogate.py
"""Checks for the few-run surrogate study (fast, CPU)."""
import math

import numpy as np
import pytest
import torch

from unified_sbi.surrogate import fisher as F
from unified_sbi.surrogate.hw import (HWConfig, HWState, branch_jacobian, energy, sample_prior,
                                      simulate, standardise, unstandardise)
from unified_sbi.surrogate.nets import Model, normalise


def test_bracket_conserves_energy_in_ideal_limit():
    """alpha, kappa, nu, D -> 0: 2D Euler for Omega. Dealiased spectral brackets conserve the
    kinetic energy up to the RK4 truncation error."""
    cfg = HWConfig(n=32, dt=0.01)
    th = np.array([[-12.0, 0.0, -12.0, -12.0, 0.2]])
    st = HWState(th, cfg, dtype=torch.float64)
    st.random_ic([0])
    st.wh = st.wh * 50                      # O(1) vorticity so the bracket matters
    e0 = energy(st.fields(), 0.2)
    f0 = st.fields()
    st.advance(200)
    e1 = energy(st.fields(), 0.2)
    assert (st.fields() - f0).norm() / f0.norm() > 0.05      # the flow did evolve
    assert abs(e1 / e0 - 1).item() < 1e-5


def test_standardise_round_trip_and_unit_variance():
    cfg = HWConfig()
    th = torch.as_tensor(sample_prior(20000, cfg, np.random.default_rng(0)))
    u = standardise(th, cfg)
    assert torch.allclose(unstandardise(u, cfg), th)
    assert (u.std(0) - 1).abs().max() < 0.02 and u.mean(0).abs().max() < 0.03


def test_restart_reproduces_trajectory_and_branch_fd_is_consistent():
    cfg = HWConfig(n=32, t_spin=60, n_save=2, batch=4)
    th = sample_prior(2, cfg, np.random.default_rng(1))
    r = simulate(th, [3, 4], cfg)
    assert r["finite"].all()
    J1, y0 = branch_jacobian(r["states"][:, 0], th, cfg, eps=0.02)
    J2, _ = branch_jacobian(r["states"][:, 0], th, cfg, eps=0.005)
    rel = lambda a, b: ((a - b).norm() / b.norm()).item()
    assert rel(y0, r["states"][:, 1]) < 1e-4
    assert rel(J1, J2) < 1e-2


def test_r_iso_minimum_positivity_and_invariance():
    torch.manual_seed(0)
    d = 5
    A = torch.randn(7, d, d, dtype=torch.float64); M = A @ A.transpose(1, 2) + 0.1 * torch.eye(d)
    B = torch.randn(7, d, d, dtype=torch.float64); G = B @ B.transpose(1, 2) + 0.1 * torch.eye(d)
    assert abs(F.r_iso(M, M).item()) < 1e-9
    assert F.r_iso(G, M).item() > 0
    U = torch.randn(d, d, dtype=torch.float64) + 3 * torch.eye(d)  # change of theta coordinates
    r1 = F.r_iso(G, M); r2 = F.r_iso(U.T @ G @ U, U.T @ M @ U)
    assert abs(r1 - r2).item() < 1e-8
    # the gauge: scaling G alone is penalised in both directions
    assert F.r_iso(4 * M, M) > 0 and F.r_iso(M / 4, M) > 0


def test_whitened_noise_covariance_and_matched_isotropic():
    torch.manual_seed(1)
    d = 3
    A = torch.randn(d, d, dtype=torch.float64); G = A @ A.T + 0.5 * torch.eye(d)
    Gb = G.expand(200000, d, d)
    delta = F.whitened_noise(Gb, 0.5)
    C = delta.T @ delta / len(delta)
    assert torch.allclose(C, 0.25 * torch.linalg.inv(G), atol=3e-3)
    iso = F.isotropic_noise_matched(Gb, 0.5)
    q_w = torch.einsum("bi,ij,bj->b", delta, G, delta).mean()
    q_i = torch.einsum("bi,ij,bj->b", iso, G, iso).mean()
    assert abs(q_w / q_i - 1) < 0.02


def test_surrogate_jacobian_matches_autograd_and_metric_of_g():
    torch.manual_seed(2)
    d, S = 5, 3
    W = torch.randn(4 * 8 * 8, d, dtype=torch.float64)
    fn = lambda x, th, lr: x + torch.tanh(th @ W.T).reshape(-1, 4, 8, 8) * lr.sum(-1)[:, None, None, None]
    x = torch.randn(S, 4, 8, 8, dtype=torch.float64); th = torch.randn(S, d, dtype=torch.float64)
    lr = torch.randn(S, 2, dtype=torch.float64)
    J = F.surrogate_jacobian(fn, x, th, lr, eps=1e-4)
    for s in range(S):
        Ja = torch.autograd.functional.jacobian(lambda t: fn(x[s:s + 1], t[None], lr[s:s + 1])[0], th[s])
        assert torch.allclose(J[s], Ja.permute(3, 0, 1, 2), atol=1e-6)
    m = Model(d, 16, K=8, width=32, depth=1, heads=2, patch=4).double()
    G = F.metric_of_g(m.g, th)
    Jg = torch.autograd.functional.jacobian(lambda t: m.g(t[None])[0], th[0])
    assert torch.allclose(G[0], Jg.T @ Jg, atol=1e-10)


def test_model_shapes_and_zero_init_residual():
    m = Model(5, 32, K=8, width=64, depth=2, heads=4, patch=4)
    s = torch.randn(3, 2, 32, 32) * 3
    x, logr = normalise(s)
    assert torch.allclose(x.pow(2).mean((-2, -1)), torch.ones(3, 2), atol=1e-5)
    y = m(x, torch.randn(3, 5), logr)
    assert y.shape == x.shape
    assert torch.allclose(y, x + (y - x))  # residual form
    th = torch.randn(3, 5)
    y0 = m(x, th, logr)
    G0 = F.metric_of_g(m.g, th)
    m.rescale_gauge(7.0)
    assert torch.allclose(m(x, th, logr), y0, atol=1e-5)          # predictions unchanged
    assert torch.allclose(F.metric_of_g(m.g, th), 49 * G0, rtol=1e-4)
    ident = Model(5, 32, identity=True, width=64, depth=1, heads=4, patch=4)
    assert ident.g.K == 5


def test_maximin_picks_far_points():
    existing = torch.zeros(1, 2)
    cand = torch.tensor([[0.1, 0.0], [5.0, 0.0], [5.1, 0.0], [-5.0, 0.0]])
    p = F.maximin(cand, existing, 2)
    assert set(p) == {1, 3} or set(p) == {2, 3}


def test_geodesic_maximin_respects_curvature():
    """Points on a circle: the chord between antipodes is 2, the geodesic pi. With one existing
    point at angle 0, geodesic maximin picks the antipode first."""
    ang = torch.linspace(0, 2 * math.pi, 201)[:-1]
    circ = torch.stack([ang.cos(), ang.sin()], 1)
    p = F.maximin_geodesic(circ[1:], circ[:1], 1, n_neighbours=4)
    assert abs(ang[1:][p[0]].item() - math.pi) < 0.05


@pytest.mark.slow
def test_study_smoke(tmp_path):
    from unified_sbi.surrogate.study import StudyConfig, load_results, run_design, run_study
    hw = HWConfig(n=16, t_spin=40, n_save=12, batch=32).to_dict()
    cfg = StudyConfig(hw=hw, pool_size=8, n_val=2, n_test=2, n_checkpoints=2, budgets=(4,),
                      seeds=(0,), arms=("A0_film", "A3_iso_wn", "A6_full"), K=6, width=32,
                      depth=1, heads=2, patch=4, steps=40, batch=4, eval_every=20, n_val_pairs=8,
                      fisher_every=5, fisher_runs=2, rollout_starts=1, rollout_h=6, flux_from=2,
                      horizons=(1, 5), design_budgets=(4,), design_candidates=32, device="cpu",
                      sim_device="cpu")
    run_study(cfg, tmp_path / "out", tmp_path / "cache", log=lambda *a: None)
    rows = load_results(tmp_path / "out")
    assert len(rows) == 3
    for r in rows:
        assert math.isfinite(r["onestep_nrmse"]) and math.isfinite(r["jac_rel_err"])
    run_design(cfg, tmp_path / "out", tmp_path / "cache", log=lambda *a: None)
    assert len(list((tmp_path / "out" / "design").rglob("metrics.json"))) == len(cfg.designs)


In [ ]:
# Checks (~1 min): energy conservation of the bracket, restart = trajectory, branch FD consistency,
# R_iso minimum/invariance, whitened-noise covariance, FD Jacobians vs autograd, exact gauge rescaling,
# chord and geodesic maximin. The end-to-end smoke test is marked slow and skipped here.
!cd {WORK} && PYTHONPATH=src python -m pytest -q -m "not slow" -p no:cacheprovider tests/test_surrogate.py

## 2. Storage and a look at the simulator
Results **and the simulation cache** go to Drive, so the simulations survive a disconnected runtime. Every finished cell is skipped on rerun.

The cell below integrates 4 runs at prior corners. It times the solver, shows that the energy saturates during spin-up, and plots the final $n$ and $\phi$ fields.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Colab Notebooks/unified-sbi/surrogate_hw')
CACHE = BASE / 'cache'; CACHE.mkdir(parents=True, exist_ok=True)

import time, math, numpy as np, matplotlib.pyplot as plt
from unified_sbi.surrogate.hw import HWConfig, simulate
h = HWConfig(n_save=2, batch=4)
lo, hi = np.array(h.lo), np.array(h.hi)
th = np.array([[hi[0], lo[1], hi[2], hi[3], hi[4]],      # adiabatic, weak drive, strong damping
               [lo[0], hi[1], lo[2], lo[3], lo[4]],      # hydrodynamic, strong drive
               (lo + hi) / 2,
               [lo[0], lo[1], hi[2], lo[3], hi[4]]])
t0 = time.time()
r = simulate(th, [0, 1, 2, 3], h, device='cuda' if torch.cuda.is_available() else 'cpu', record_spinup_every=250)
print(f'{time.time() - t0:.1f} s for 4 runs x {int((h.t_spin + h.n_save * h.save_dt) / h.dt)} steps; finite: {r["finite"].tolist()}')
fig, ax = plt.subplots(1, 9, figsize=(18, 2.3), constrained_layout=True)
tt = np.arange(1, r['spinup_energy'].shape[1] + 1) * 250 * h.dt
for i in range(4):
    ax[0].semilogy(tt, r['spinup_energy'][i], lw=2)
    ax[1 + 2 * i].imshow(r['states'][i, -1, 0], cmap='RdBu_r'); ax[1 + 2 * i].set_title(f'run {i}: n', fontsize=8)
    ax[2 + 2 * i].imshow(r['states'][i, -1, 1], cmap='RdBu_r'); ax[2 + 2 * i].set_title(f'run {i}: phi', fontsize=8)
ax[0].set_title('spin-up energy', fontsize=8); ax[0].set_xlabel('t', fontsize=8)
for a in ax[1:]: a.axis('off')
plt.show()

## 3. Configure and run
- **`smoke`**: a few minutes. It runs the whole pipeline (simulation, training, evaluation, plots) on a small pool and prints step timings. Run it first.
- **`decisive`**: A0/A1/A2/A4 at $N_\theta\in\{25,50,100,200\}$ with 2 seeds.
- **`full`**: all seven arms with 3 seeds.

**Cost.**
- Simulating the pool (256 + 16 + 48 runs, ≈18.5k solver steps each), plus 4 checkpoints × 11 branch restarts per pool and test run, should take minutes on an A100. The data are cached on Drive.
- Training is 8000 steps per cell. The Fisher refreshes add roughly 10–20% to the R arms, and the Jacobian arms (A4, A6) cost about 2× per step.
- These estimates are untested on Colab: check the `smoke` timings first.

**Settings.** Changing any training or data setting needs a new results folder; the runner refuses to mix configurations. `arms`, `seeds`, `budgets` and `design_budgets` can be changed freely, so you can add arms to a finished folder later.

In [ ]:
from unified_sbi.surrogate.study import ARMS, DECISIVE, StudyConfig, run_study, run_design
from unified_sbi.surrogate.hw import HWConfig

MODE = 'smoke'              # 'smoke' | 'decisive' | 'full'
hw = HWConfig(batch=128).to_dict()
if MODE == 'smoke':
    hw.update(n_save=60)
    cfg = StudyConfig(hw=hw, pool_size=32, n_val=8, n_test=16, budgets=(8, 24), seeds=(0,),
                      arms=DECISIVE, steps=600, eval_every=200, width=128, depth=4, heads=4,
                      design_budgets=(16,), design_candidates=512)
elif MODE == 'decisive':
    cfg = StudyConfig(hw=hw, arms=DECISIVE, seeds=(0, 1))
else:
    cfg = StudyConfig(hw=hw, arms=tuple(ARMS), seeds=(0, 1, 2))
OUT = BASE / MODE
print(OUT)
data = run_study(cfg, OUT, CACHE)       # order: seed -> budget -> arm; finished cells are skipped

## 4. Figures and tables
The paired table gives seed-level differences against A0 (positive = better than the FiLM baseline), with standard errors over seeds.

In [ ]:
from IPython.display import Image, display
import pandas as pd
!cd {WORK} && PYTHONPATH=src python scripts/plot_surrogate.py --results "{OUT}"
for f in ['scaling.png', 'sensitivity_by_param.png']:
    if (OUT / 'figures' / f).exists(): display(Image(filename=str(OUT / 'figures' / f)))
p = OUT / 'tables' / 'paired_vs_A0.csv'
if p.exists():
    t = pd.read_csv(p)
    cols = ['arm', 'budget'] + [c for c in t.columns if c.endswith('_gain') or c.endswith('_se')]
    display(t[cols].round(4))

## 5. Design experiment (P5, optional)
For each seed and each $N$ in `design_budgets`, this trains the design arm (`A3_iso_wn` by default) on $N/2$ random runs. It then adds $N/2$ runs chosen in one of five ways and retrains on all $N$:
- `random`, the next pool runs;
- `lhs`, a Latin hypercube;
- `maximin_theta`, maximin in standardised θ;
- `maximin_g`, maximin on the chord distance in the learned $g$-space;
- `maximin_geo`, maximin on the graph-geodesic distance along the learned manifold.

New runs are simulated and cached.

In [ ]:
RUN_DESIGN = False
if RUN_DESIGN:
    import dataclasses
    dcfg = dataclasses.replace(cfg, design_budgets=(50, 100) if MODE != 'smoke' else (16,))
    run_design(dcfg, OUT, CACHE, data=data)
    !cd {WORK} && PYTHONPATH=src python scripts/plot_surrogate.py --results "{OUT}"
    display(Image(filename=str(OUT / 'figures' / 'design.png')))
    display(pd.read_csv(OUT / 'tables' / 'design.csv').round(4))

## 6. Export
This zips the metrics, tables, figures and per-run arrays. Model weights and the simulation cache are left out.

In [ ]:
import shutil, tempfile
from google.colab import files
with tempfile.TemporaryDirectory() as tmp:
    for p in OUT.rglob('*'):
        if p.suffix in {'.json', '.csv', '.png', '.npz'}:
            q = Path(tmp) / p.relative_to(OUT); q.parent.mkdir(parents=True, exist_ok=True); shutil.copy(p, q)
    archive = shutil.make_archive(f'/content/surrogate_{MODE}', 'zip', tmp)
files.download(archive)